# 07 — Détection avec le modèle entraîné (inférence)

**Équipe 36 — lot G — U-Net 2D.** Ce notebook *utilise* les modèles entraînés dans
les notebooks 03 (socle) et 04 (variante) : vous lui donnez une image ou un scanner,
il renvoie les masques d'hémorragie prédits (seuil 0,5) et le détail lésion par
lésion (volume, classe de taille), exactement comme à l'entraînement.

**Entrées acceptées :**
- un scanner NIfTI `.nii` / `.nii.gz` (toutes les coupes sont prédites, le masque 3D est sauvé) ;
- une image 2D `.png` / `.jpg` / `.jpeg` / `.bmp` / `.tif` (une coupe, supposée déjà
  fenêtrée [-40, 120] UH : intensité 0–255 → remappée sur [-1, +1] par 2·(v/255) − 1,
  ce qui correspond exactement à la normalisation (valeur − 40) / 80).

**À régler dans la cellule 2 :** `CHEMIN_ENTREE` (votre image/scanner) et `CONFIG`
(`'socle'` ou `'variante'`). Tout le reste est automatique.

> Notebook d'inférence hors protocole commun : il ne participe pas au tableau
> comparatif des équipes, il sert à montrer et utiliser le modèle.

In [ ]:
# --- Modele : architecture identique a l'entraînement --------------------
import json
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage

CT_MIN_HU, CT_MAX_HU = -40.0, 120.0
SEUIL = 0.5
CONNECTIVITE = np.ones((3, 3, 3), dtype=bool)   # 26-voisinage, comme le projet
VOLUME_MIN_ML = 0.01
CLASSES = [(0.1, "< 0,1 mL"), (1.0, "0,1 à 1 mL"), (5.0, "1 à 5 mL"), (float("inf"), "> 5 mL")]
SPACING_BHSD = (0.48828125, 0.48828125, 6.0)    # mm, BHSD ; utilisé pour les images 2D

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch),
                                 nn.ReLU(inplace=True), nn.Conv2d(out_ch, out_ch, 3, padding=1),
                                 nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.net(x)

class UNet2D(nn.Module):
    """U-Net 2D a 4 niveaux — même architecture que les notebooks 03/04."""
    def __init__(self, in_ch=1, out_ch=1, features=(32, 64, 128, 256)):
        super().__init__()
        self.downs = nn.ModuleList(); self.ups = nn.ModuleList(); self.pool = nn.MaxPool2d(2)
        ch = in_ch
        for f in features:
            self.downs.append(DoubleConv(ch, f)); ch = f
        self.bottleneck = DoubleConv(ch, ch * 2); ch = ch * 2
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(ch, f, 2, stride=2))
            self.ups.append(DoubleConv(f * 2, f)); ch = f
        self.final = nn.Conv2d(features[0], out_ch, 1)
    def forward(self, x):
        skip = []
        for down in self.downs:
            x = down(x); skip.append(x); x = self.pool(x)
        x = self.bottleneck(x); skip = skip[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x); s = skip[i // 2]
            if x.shape[-2:] != s.shape[-2:]:
                x = F.interpolate(x, size=s.shape[-2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, s], dim=1); x = self.ups[i + 1](x)
        return self.final(x)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Modeles : dossier local en local, dataset monte sur Kaggle. Le montage peut
# inclure un prefixe (ex. /kaggle/input/datasets/aymencharafedn/...) : on cherche
# recursivement le fichier de poids.
def trouver_modeles():
    if Path('modeles/model_socle.pt').is_file():
        return Path('modeles')
    if Path('/kaggle/input').is_dir():
        _trouve = next(Path('/kaggle/input').rglob('model_socle.pt'), None)
        if _trouve is not None:
            return _trouve.parent
    raise FileNotFoundError('model_socle.pt introuvable (local ou /kaggle/input)')

MODELES_DIR = trouver_modeles()
print('Dossier des modeles :', MODELES_DIR.resolve() if MODELES_DIR.is_dir() else MODELES_DIR)
print('Dispositif :', DEVICE)

In [ ]:
# --- A regler : l'image a analyser et le modele a utiliser ----------------
CHEMIN_ENTREE = 'data/label_192/images/ID_0237f3c9_ID_40015688b9.nii.gz'  # <-- VOTRE image/scanner
CONFIG = 'socle'          # 'socle' ou 'variante'
# --------------------------------------------------------------------------

assert CONFIG in ('socle', 'variante')
CHEMIN_ENTREE = Path(CHEMIN_ENTREE)
if not CHEMIN_ENTREE.is_file() and Path('/kaggle/input').is_dir():
    # Kaggle : cherche le fichier dans les datasets montes. Les .nii.gz y sont
    # decompresses en .nii : on teste les deux noms. Le meme nom existe sous
    # images/ ET ground truths/ : on garde la copie sous images/.
    _noms = [CHEMIN_ENTREE.name]
    if CHEMIN_ENTREE.name.endswith('.nii.gz'):
        _noms.append(CHEMIN_ENTREE.name[:-3])
    _candidats = []
    for _nom in _noms:
        _candidats = list(Path('/kaggle/input').rglob(_nom))
        if _candidats:
            break
    _images = [p for p in _candidats
               if 'images' in str(p).lower() and 'ground' not in str(p).lower()]
    if _images or _candidats:
        CHEMIN_ENTREE = (_images or sorted(_candidats))[0]
assert CHEMIN_ENTREE.is_file(), f'Fichier introuvable : {CHEMIN_ENTREE}'

modele = UNet2D().to(DEVICE)
chemin_pt = MODELES_DIR / f'model_{CONFIG}.pt'
modele.load_state_dict(torch.load(chemin_pt, map_location=DEVICE))
modele.eval()
print(f'Modele {CONFIG} charge : {chemin_pt} ({sum(p.numel() for p in modele.parameters()):,} parametres)')
print('Entree :', CHEMIN_ENTREE)

In [ ]:
# --- Chargement de l'entree -> liste de coupes normalisees ---------------
EXT_2D = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def charger_entree(chemin):
    """Renvoie (coupes, spacing_zyx, origine).
    coupes : liste de tableaux 2D normalises (v-40)/80, 512x512.
    origine : 'nifti_hu' (scanner en UH) ou 'image_2d' (deja fenetre, 0-255).
    """
    ext = chemin.suffix.lower()
    if ext in ('.nii', '.gz') or chemin.name.endswith('.nii.gz'):
        img = nib.load(chemin)
        vol = np.asarray(img.dataobj, dtype=np.float32)
        zooms = img.header.get_zooms()[:3]
        spacing = tuple(float(z) if z and z > 0 else s for z, s in zip(zooms, SPACING_BHSD))
        coupes = []
        for z in range(vol.shape[2]):
            c = np.clip(vol[:, :, z], CT_MIN_HU, CT_MAX_HU)
            coupes.append((c - 40.0) / 80.0)
        return coupes, spacing, 'nifti_hu'
    if ext in EXT_2D:
        im = plt.imread(chemin)
        im = np.asarray(im, dtype=np.float32)
        if im.ndim == 3:
            im = im[..., :3].mean(axis=2)
        if im.max() > 1.5:
            im = im / 255.0
        # image 8 bits deja fenetree : 0 <-> -40 UH, 255 <-> +120 UH. La
        # normalisation d'entraînement (v-40)/80 couvre [-1, +1] d'ou 2x-1.
        im = 2.0 * im - 1.0
        coupes = [im]
        return coupes, SPACING_BHSD, 'image_2d'
    raise ValueError(f'Format non pris en charge : {ext}')

def a_512(coupe):
    if coupe.shape == (512, 512):
        return coupe
    from scipy.ndimage import zoom
    return zoom(coupe, (512 / coupe.shape[0], 512 / coupe.shape[1]), order=1)

coupes, spacing, origine = charger_entree(CHEMIN_ENTREE)
coupes = [a_512(c) for c in coupes]
voxel_ml = float(np.prod(spacing)) / 1000.0
print(f'{CHEMIN_ENTREE.name} : {len(coupes)} coupe(s), origine {origine}, '
      f'voxel {spacing[0]:.3f} x {spacing[1]:.3f} x {spacing[2]:.1f} mm '
      f'({voxel_ml:.4f} mL/voxel)')

In [ ]:
# --- Prediction coupes par coupes (batch 8, comme a l'entraînement) ------
probs_vol = np.zeros((512, 512, len(coupes)), dtype=np.float32)
with torch.no_grad():
    for i0 in range(0, len(coupes), 8):
        lot = np.stack([coupes[i] for i in range(i0, min(i0 + 8, len(coupes)))])
        tenseur = torch.from_numpy(lot[:, None]).float().to(DEVICE)
        logits = modele(tenseur)
        probs = torch.sigmoid(logits)[:, 0].cpu().numpy()
        # le lot sort en (batch, X, Y) -> on le remet en (X, Y, Z)
        probs_vol[:, :, i0:i0 + probs.shape[0]] = probs.transpose(1, 2, 0)
masque = probs_vol > SEUIL
n_vox = int(masque.sum())
print(f'Masque predit : {n_vox:,} voxels positifs (seuil {SEUIL})')

In [ ]:
# --- Lesions : decompte, volumes, classes de taille ----------------------
labels, n_les = ndimage.label(masque, structure=CONNECTIVITE)
lignes = []
for k in range(1, n_les + 1):
    m = labels == k
    volume = float(m.sum()) * voxel_ml
    if volume < VOLUME_MIN_ML:
        continue
    zc = int(np.round(np.argwhere(m)[:, 2].mean()))
    classe = next(nom for seuil, nom in CLASSES if volume < seuil)
    lignes.append({'lesion': len(lignes) + 1, 'coupe': zc,
                   'volume_ml': round(volume, 3), 'classe_taille': classe,
                   'voxels': int(m.sum()),
                   'confiance': round(float(probs_vol[m].mean()), 3)})
table = pd.DataFrame(lignes)
n_petites = int((table['volume_ml'] < 5.0).sum()) if len(table) else 0
print()
print(f'>>> {len(table)} hemorragie(s) detectee(s) '
      f'(dont {n_petites} de moins de 5 mL) — modele {CONFIG}')
if len(table):
    print(table.to_string(index=False))
else:
    print('Aucune hemorragie detectee au seuil', SEUIL)

SORTIE = Path('resultats_detection') / (CHEMIN_ENTREE.stem + '_' + CONFIG)
SORTIE.mkdir(parents=True, exist_ok=True)
table.to_csv(SORTIE / 'lesions_detectees.csv', index=False)

In [ ]:
# --- Figures + masques de sortie -----------------------------------------
def tracer_coupe(z, ax):
    ax.imshow(coupes[z], cmap='gray', origin='lower')
    m2d = masque[:, :, z]
    if m2d.any():
        ax.contour(m2d, levels=[0.5], colors='#d62728', linewidths=1.4)
        ax.imshow(np.ma.masked_where(~m2d, m2d), cmap='Reds', alpha=0.35, origin='lower')

coupes_detectees = sorted({l['coupe'] for l in lignes}) if lignes else []
if coupes_detectees:
    colonnes = min(4, len(coupes_detectees))
    fig, axes = plt.subplots(1, colonnes, figsize=(4.2 * colonnes, 4.4), squeeze=False)
    for ax, z in zip(axes[0], coupes_detectees[:colonnes]):
        tracer_coupe(z, ax)
        vols = [l['volume_ml'] for l in lignes if l['coupe'] == z]
        ax.set_title(f'coupe {z} — {sum(vols):.2f} mL', fontsize=10)
        ax.axis('off')
    fig.suptitle(f'Hemorragies detectees — {CONFIG}, seuil {SEUIL}', fontsize=12)
    fig.tight_layout()
    fig.savefig(SORTIE / 'detection_overlay.png', dpi=180, bbox_inches='tight')
    plt.show()
else:
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.imshow(coupes[len(coupes) // 2], cmap='gray', origin='lower')
    ax.set_title('Aucune hemorragie detectee'); ax.axis('off')
    fig.savefig(SORTIE / 'detection_overlay.png', dpi=180, bbox_inches='tight')
    plt.show()

# Masque : NIfTI (meme geometrie) pour un scanner, PNG pour une image 2D
if origine == 'nifti_hu':
    img0 = nib.load(CHEMIN_ENTREE)
    nib.save(nib.Nifti1Image(masque.astype(np.uint8), img0.affine), str(SORTIE / 'masque_predit.nii.gz'))
else:
    plt.imsave(SORTIE / 'masque_predit.png', masque[:, :, 0], cmap='gray')
print('Sorties dans :', SORTIE.resolve())
for f in sorted(SORTIE.iterdir()):
    print('  -', f.name)

## Notes et limites

- **Même protocole qu'à l'entraînement** : fenêtre [-40, 120] UH, normalisation (valeur − 40) / 80
  (couvre [-1, +1]), coupes 512 × 512, seuil 0,5, lot de 8 en inférence. Sur une image 2D 8 bits
  supposée déjà fenêtrée : 2·(v/255) − 1 restitue la même normalisation.
- **Volumes** : calculés avec l'espacement du NIfTI lu dans le fichier ; pour une image 2D,
  l'espacement BHSD (0,488 × 0,488 × 6 mm) est utilisé par défaut — volumes indicatifs.
- Les lésions de moins de 0,01 mL sont ignorées (comme dans toute la chaîne du projet).
- Performance attendue (jeu de test officiel, 864 lésions) : sensibilité 41,9 % (socle) /
  44,1 % (variante) toutes lésions, 96,5–98,8 % au-dessus de 5 mL — les petites lésions
  restent difficiles, c'est précisément l'objet du lot G.